# MHD in 3D

`MHD3D` solves ideal and resistive MHD in a box, on the patches of a 3D patchwork
hierarchy. It is `MHD2D` without the plane symmetry: the same MUSCL–HLLD fluxes along each
axis, Heun's method and refluxing, with the vector potential on the cell edges instead of
$A_z$ on the nodes. The flux through a face is the circulation of $\mathbf A$ around it,
so $\nabla\cdot\mathbf B = 0$ in every cell to round-off, across refined patches too.

In [ ]:
import numpy as np
import plasma_plots

from carafe import MHD2D, MHD3D

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex");

## An oblique Alfvén wave

A circularly polarized Alfvén wave travels along $(1, 2, 2)/3$, oblique to all three axes.
After one period the exact solution is the initial state; the error falls at second order
with the unlimited reconstruction.

In [ ]:
errors = []
for cells in (4, 8, 16):
    model, exact = MHD3D.alfven_wave(cells, limiter="none")
    model.run(model.period)
    by = model.dataset()["magnetic_field_y"].values
    errors.append(np.mean(np.abs(by - exact["magnetic_field_y"].values)))
    print(
        f"{cells:3d} cells: error {errors[-1]:.2e}, max |div B| dx {model.divergence():.1e}"
    )
print("orders:", np.round(np.log2(np.array(errors[:-1]) / errors[1:]), 2))

## The 2D limit

A state that does not depend on $z$ stays the 2D solution: the Orszag–Tang vortex in a slab
of two layers follows `MHD2D` step by step.

In [ ]:
slab = MHD3D.orszag_tang(32, layers=2)
plane = MHD2D.orszag_tang(32)
for _ in range(20):
    dt = plane.stable_step()
    plane.step(dt)
    slab.step(dt)
difference = np.abs(
    slab.dataset().density.values[:, :, 0] - plane.dataset().density.values
).max()
print(f"largest density difference after 20 steps: {difference:.1e}")

## A magnetized blast wave

A sphere of high pressure in a gas at plasma beta 0.2, the field along the diagonal
$(1, 1, 1)/\sqrt 3$ (Gardiner and Stone 2008). The field resists compression across it, so
the blast runs along the field lines. Mass and energy stay constant in the periodic box.

In [ ]:
blast = MHD3D.blast_wave(32)
start = blast.totals()
blast.run(0.1)
end = blast.totals()
print({name: f"{end[name] - start[name]:.1e}" for name in start})
print(f"max |div B| dx: {blast.divergence():.1e}")
state = blast.dataset()
state.pressure.sel(z=0.0, method="nearest").plasma.plot.slice(x="x", y="y")

The magnetic pressure in the same plane: the field lines are pushed aside across the
field and bunched at the front of the shell.

In [ ]:
state.magnetic_pressure.sel(z=0.0, method="nearest").plasma.plot.slice(x="x", y="y")